# Messages Analyst
Loads decoded messages and runs analysis.

In [41]:
%run messages_decoder.ipynb

In [42]:
data = decode_messages()

Level-2 OK — 44422 messages, 37 participants.
Level-1 OK — 37 names restored.


In [43]:
DISPLAY_WHITELIST: set[str] = {
    "Duy Lê",
    "Hồng Nhung",
    "Huy Nguyễn",
    "Tri Phan",
    "Linh Tran Hoang",
    "Nhung Tran",
    "Hoàng Hữu Phong",
    "Đinh Thị Nghĩa",
    "Khổng Vũ Minh Thái",
    "Tu Dao Pham",
    "Lê Văn Hữu Thịnh",
    "Jun Ng",
    "Mau Dinh Nguyen",
    "Thắng Quốc Phạm",
    "Tử Kỳ",
    "Dong Mai Hoa",
    "Himiko Tnk",
    "Huỳnh Khang Ninh",
    "May Ca",
    "Bee",
    "Phương Hạ",
    "Lan Hương",
    "Hoàii Thu",
    "Anh Thư",
    "Bế Minh Nhật",
    "Đặng Anh Vũ",
}

def display_name(name: str) -> str:
    if name.startswith('gAAAAA') or name in DISPLAY_WHITELIST:
        return name
    return '***' # + name for debugging

---
## Message count per person
Sorted high → low, with cumulative % of total.

In [44]:
from collections import Counter

counts = Counter(
    msg['sender_name']
    for msg in data['messages']
    if 'sender_name' in msg
)

total = sum(counts.values())
ranked = counts.most_common()

output = []

output.append("## Message count per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Messages':>9} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 72)

cumulative = 0
for rank, (name, count) in enumerate(ranked, start=1):
    share = count / total * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {count:>9,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Reactions given per person
How many reactions each person has given, sorted high → low.

In [45]:
reactions_given = Counter(
    react['actor']
    for msg in data['messages']
    for react in msg.get('reactions', [])
    if 'actor' in react
)

total_given = sum(reactions_given.values())
ranked_given = reactions_given.most_common()

output.append("\n## Reactions given per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Given':>7} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 70)

cumulative = 0
for rank, (name, count) in enumerate(ranked_given, start=1):
    share = count / total_given * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {count:>7,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Reactions received per person
How many reactions each person's messages have received, sorted high → low.

In [46]:
reactions_received = Counter(
    msg['sender_name']
    for msg in data['messages']
    if 'sender_name' in msg
    for _ in msg.get('reactions', [])
)

total_received = sum(reactions_received.values())
ranked_received = reactions_received.most_common()

output.append("\n## Reactions received per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Received':>9} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 72)

cumulative = 0
for rank, (name, count) in enumerate(ranked_received, start=1):
    share = count / total_received * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {count:>9,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Reactions received / messages sent ratio
Higher ratio = messages tend to generate more reactions. Only includes people with ≥ 10 messages.

In [47]:
MIN_MESSAGES = 10

all_names = set(counts.keys()) | set(reactions_received.keys())
ratios = [
    (name, reactions_received.get(name, 0), counts.get(name, 0),
     reactions_received.get(name, 0) / counts[name])
    for name in all_names
    if counts.get(name, 0) >= MIN_MESSAGES
]
ratios.sort(key=lambda x: x[3], reverse=True)

output.append("\n## Reactions received / messages sent ratio")
output.append(f"{'Rank':<5} {'Name':<35} {'Received':>9} {'Messages':>9} {'Ratio':>7}")
output.append('-' * 70)

for rank, (name, received, msgs, ratio) in enumerate(ratios, start=1):
    output.append(f"{rank:<5} {display_name(name):<35} {received:>9,} {msgs:>9,} {ratio:>7.3f}")

In [48]:
import os

out_path = os.path.normpath(os.path.join(os.getcwd(), '..', 'Data', 'Result', 'analysis.txt'))
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(output))

print(f"Written to {out_path}")

Written to /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/Result/analysis.txt
